[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fw-ai/cookbook/blob/main/training/case-studies/sft_prompt_router/prompt_router_dedicated.ipynb)

# SFT: prompt-routing classifier — pure Python SDK

Fine-tune a small model to be a **prompt router**: given a user request, predict — in one
JSON object — whether it should be handled by a `small model` or escalated to a `big model`,
plus the features that drive that decision (complexity, whether it needs math / code / reasoning).

This is a **multi-field text classification** task done entirely with the **Fireworks Python
SDK** (`fireworks-ai`): build the data, evaluate the base model, run a managed LoRA SFT job,
deploy, and evaluate again — measuring the lift per field.

**Dataset:** [`SupraLabs/Prompt-Routing-Dataset`](https://huggingface.co/datasets/SupraLabs/Prompt-Routing-Dataset)
(992 rows; each has `routing_choice`, `complexity_score`, `math_task`, `coding_task`,
`requires_reasoning`). The routing target follows a deterministic rule baked into the data
(`complexity >= 3 OR code OR math -> big model`).

**Fair comparison.** We give the **base** model the *rich* prompt — the routing rule spelled
out, i.e. best-effort prompt engineering, how you'd really use a base model. The **tuned**
model gets only a *lean* prompt (output schema, no rule) because it learned the policy during
SFT. So it's prompt-engineered base vs a fine-tuned model on a short prompt: if the tuned
model matches or beats the well-prompted base, the fine-tune earns its keep (shorter/cheaper
prompt, no policy to maintain by hand).

**"Customer problem":** cheap, local edge routing — keep easy chat on a small model, escalate
hard logic/code/math to a frontier model, without calling the big model just to decide.

> Everything runs in this notebook (no helper scripts). Train + deploy cells provision real
> GPU and **cost money**. Defaults are smoke-sized; scale `N_TRAIN` / `N_EVAL` up for real signal.

## 1. Setup & credentials

Imports, repo-root `.env` loading, the Fireworks SDK client, and shared poll/upload helpers.

In [1]:
import json
import os
import sys
import time
import uuid
from pathlib import Path

import dotenv

dotenv.load_dotenv(dotenv.find_dotenv(usecwd=True), override=True)  # repo-root .env

# Put repo root + common/ on the path so the shared eval helper (ep_eval) imports.
_REPO_ROOT = Path.cwd()
while _REPO_ROOT != _REPO_ROOT.parent and not (_REPO_ROOT / "common" / "ep_eval.py").exists():
    _REPO_ROOT = _REPO_ROOT.parent
for _p in (str(_REPO_ROOT), str(_REPO_ROOT / "common")):
    if _p not in sys.path:
        sys.path.insert(0, _p)

from fireworks import Fireworks, file_from_path

assert os.getenv("FIREWORKS_API_KEY"), "FIREWORKS_API_KEY missing from .env"
ACCOUNT_ID = os.getenv("FIREWORKS_ACCOUNT_ID", "").replace("accounts/", "", 1)
assert ACCOUNT_ID, "FIREWORKS_ACCOUNT_ID missing from .env"

client = Fireworks()  # reads FIREWORKS_API_KEY + FIREWORKS_ACCOUNT_ID from env


def wait_until(get_fn, ok, bad, label, every=30, tries=240):
    obj = None
    for i in range(tries):
        obj = get_fn()
        st = getattr(obj, "state", None)
        print(f"[{label}] {i + 1:03d} state={st}")
        if st in ok:
            return obj
        if st in bad:
            raise RuntimeError(f"{label} failed: state={st}")
        time.sleep(every)
    raise TimeoutError(f"{label} did not finish after {tries} polls")


def upload_dataset(ds_id, path, fmt="CHAT"):
    n_examples = sum(1 for line in open(path, encoding="utf-8") if line.strip())
    client.datasets.create(dataset_id=ds_id, dataset={"format": fmt, "example_count": n_examples})
    client.datasets.upload(dataset_id=ds_id, file=file_from_path(path))
    wait_until(lambda: client.datasets.get(ds_id), {"READY"}, {"STATE_UNSPECIFIED"},
               "dataset", every=3, tries=200)
    return f"accounts/{ACCOUNT_ID}/datasets/{ds_id}"


print("SDK client ready. account:", ACCOUNT_ID)

SDK client ready. account: pyroworks


## 2. Configuration

The label schema (what the classifier predicts), sample sizes, and SFT knobs — the main
things you'll edit. `LABELS` defines the output keys/allowed values (the schema is shown to
the model), but the **routing policy is deliberately hidden** from the prompt — the base
model has to guess the boundary and the fine-tune is what teaches it, so the lift is real.
`route` is emitted last (features first), and its accuracy is the headline metric.

In [2]:
# CONFIG
# qwen3p5-9b: the SAME base the serverless notebook trains, so managed (dedicated) vs
# serverless is an apples-to-apples comparison. It's deployed on-demand for eval, scored,
# and torn down (see eval_deployed below) — same pattern as the tuned model.
BASE_MODEL = "accounts/fireworks/models/qwen3p5-9b"
HF_DATASET = "SupraLabs/Prompt-Routing-Dataset"

# Fields the classifier predicts (a subset of the dataset columns) + their allowed values.
# `source` is the dataset column each field is derived from. Order matters: the model
# emits the features FIRST and `route` LAST, so it "computes" the decision inputs before
# committing to the route (the dataset authors' recommended multi-task ordering).
LABELS = {
    "complexity": {"source": "complexity_score",    "values": [1, 2, 3, 4, 5]},
    "math":       {"source": "math_task",           "values": [True, False]},
    "code":       {"source": "coding_task",         "values": [True, False]},
    "reasoning":  {"source": "requires_reasoning",  "values": [True, False]},
    "route":      {"source": "routing_choice",      "values": ["small model", "big model"]},
}
FIELDS = list(LABELS)
PRIMARY_FIELD = "route"   # the actual decision; headline metric

# Sample sizes (dataset has 992 rows; bump for more signal).
N_TRAIN = 600
N_EVAL = 60
SEED = 0

# SFT knobs
EPOCHS = 3
LORA_RANK = 8
LEARNING_RATE = 1e-4
BATCH_SIZE_SAMPLES = 0   # 0 = token-based batching (>0 can overflow the token budget)
ACCELERATOR_TYPE = "NVIDIA_H200_141GB"   # SDK deploy needs an explicit accelerator
MAX_TOKENS = 1536        # room for the JSON (thinking is disabled via /no_think in the prompt)

# --- Cost / speed accounting (dedicated-vs-serverless comparison) ---
# Dedicated bills per GPU-hour while the deployment is live. Set this to your account's
# on-demand $/GPU-hr for ACCELERATOR_TYPE (see fireworks.ai/pricing). Placeholder value.
H200_HOURLY_USD = 10.00
DEPLOY_REPLICAS = 1
# Serverless per-token rates for qwen3p5-9b (serverless.md), shown as a token-equivalent
# reference next to the dedicated GPU-hour cost.
SERVERLESS_PREFILL_PER_1M = 0.44
SERVERLESS_SAMPLE_PER_1M = 1.33

TRAIN_FILE = "./prompt_routing_train.jsonl"   # regenerated below (gitignored)
DATASET_ID = f"route-sft-{uuid.uuid4().hex[:6]}"
OUTPUT_MODEL_ID = f"qwen3p5-9b-router-sdk-{uuid.uuid4().hex[:6]}"
DEPLOYMENT_ID = f"router-sdk-{uuid.uuid4().hex[:6]}"
print("dataset_id:", DATASET_ID, "| output_model_id:", OUTPUT_MODEL_ID)

dataset_id: route-sft-a73198 | output_model_id: qwen3p5-9b-router-sdk-4973aa


## 3. Build the dataset

Download the routing dataset from HuggingFace, turn each row into a chat example
(system = the **lean** prompt, user = the prompt, assistant = the gold JSON label), then
split into train / eval and write the training file. We define two system prompts here:
`LEAN_PROMPT` (schema only — used for training and the tuned-model eval) and `RICH_PROMPT`
(lean + the routing rule — used to give the base model its best shot).

In [3]:
from datasets import load_dataset


def _fmt_values(vals):
    return ", ".join(json.dumps(v) for v in vals)


# Two prompts, for a FAIR comparison of how each model is realistically used:
#   * LEAN_PROMPT  — just the output schema, no routing policy. Used to TRAIN and to eval
#     the tuned model: it has internalized the policy, so it needs no hand-holding.
#   * RICH_PROMPT  — LEAN plus the explicit routing rule. This is what a careful user would
#     hand the *base* model to get the best out of it (prompt engineering).
# We eval base with RICH and tuned with LEAN: prompt-engineered base vs fine-tuned model on a
# short prompt. If tuned (lean) matches/beats base (rich), the fine-tune earns its keep —
# same quality with a shorter, cheaper prompt and no policy to maintain by hand.
_SCHEMA = (
    "Respond with ONLY a compact JSON object (no prose, no code fence) with exactly these "
    "keys, in this order:\n"
    f"- \"complexity\": integer {_fmt_values(LABELS['complexity']['values'])} (1=trivial, 5=advanced)\n"
    f"- \"math\": {_fmt_values(LABELS['math']['values'])} (needs symbolic math / proofs / multi-step word math)\n"
    f"- \"code\": {_fmt_values(LABELS['code']['values'])} (needs code generation / debugging)\n"
    f"- \"reasoning\": {_fmt_values(LABELS['reasoning']['values'])} (needs multi-step reasoning)\n"
    f"- \"route\": one of {_fmt_values(LABELS['route']['values'])}\n\n"
    'Example: {"complexity": 3, "math": true, "code": false, "reasoning": true, "route": "big model"}'
)
_INTRO = (
    "You are a prompt-routing classifier for an LLM serving stack. Given a user prompt, "
    "analyze the task and decide whether a small local model can handle it or whether it "
    "must be escalated to a big frontier model.\n\n"
)
_RULE = (
    "Routing policy: choose \"big model\" if complexity >= 3 OR the task needs code OR the "
    "task needs math; otherwise choose \"small model\".\n\n"
)
# Qwen3 is a hybrid-thinking model; "/no_think" disables the reasoning channel so it emits
# the JSON directly (clean output, and SFT on terse JSON doesn't fight a reasoning format).
_NOTHINK = "\n/no_think"

LEAN_PROMPT = _INTRO + _SCHEMA + _NOTHINK           # tuned model (learned policy) + training target
RICH_PROMPT = _INTRO + _RULE + _SCHEMA + _NOTHINK   # best-effort prompting for the base model


def gold_label(ex) -> dict:
    # Keys ordered to match LABELS: features first, route last.
    return {
        "complexity": int(ex["complexity_score"]),
        "math": bool(ex["math_task"]),
        "code": bool(ex["coding_task"]),
        "reasoning": bool(ex["requires_reasoning"]),
        "route": ex["routing_choice"],
    }


def to_chat(ex) -> dict:
    # Train on the LEAN prompt so the tuned model works without the policy spelled out.
    return {
        "messages": [
            {"role": "system", "content": LEAN_PROMPT},
            {"role": "user", "content": ex["prompt"]},
            {"role": "assistant", "content": json.dumps(gold_label(ex))},
        ]
    }


ds = load_dataset(HF_DATASET, split="train").shuffle(seed=SEED)
train_ex = [ds[i] for i in range(N_TRAIN)]
eval_ex = [ds[i] for i in range(N_TRAIN, N_TRAIN + N_EVAL)]

with open(TRAIN_FILE, "w", encoding="utf-8") as f:
    for ex in train_ex:
        f.write(json.dumps(to_chat(ex)) + "\n")

print(f"wrote {len(train_ex)} train rows -> {TRAIN_FILE}; held out {len(eval_ex)} for eval")
print("\nExample training row:")
print(json.dumps(to_chat(train_ex[0]), indent=2)[:900])

/Users/sinan/cookbook-casestudies/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


wrote 600 train rows -> ./prompt_routing_train.jsonl; held out 60 for eval

Example training row:
{
  "messages": [
    {
      "role": "system",
      "content": "You are a prompt-routing classifier for an LLM serving stack. Given a user prompt, analyze the task and decide whether a small local model can handle it or whether it must be escalated to a big frontier model.\n\nRespond with ONLY a compact JSON object (no prose, no code fence) with exactly these keys, in this order:\n- \"complexity\": integer 1, 2, 3, 4, 5 (1=trivial, 5=advanced)\n- \"math\": true, false (needs symbolic math / proofs / multi-step word math)\n- \"code\": true, false (needs code generation / debugging)\n- \"reasoning\": true, false (needs multi-step reasoning)\n- \"route\": one of \"small model\", \"big model\"\n\nExample: {\"complexity\": 3, \"math\": true, \"code\": false, \"reasoning\": true, \"route\": \"big model\"}\n/no_think"
    },
    {
      "role": "user",
      "content": "If the deli runs out o


## 4. Evaluation helpers

We score both models through **eval-protocol** (`single_turn_eval` in `common/ep_eval.py`).
The reward parses the model's JSON and scores it **per field** against the gold label; the
row's score is the fraction of the five fields it got right. `run_eval` then aggregates
overall exact-match (all five correct) and per-field accuracy so we can see where the lift is.

In [4]:
from ep_eval import single_turn_eval, EvaluationRow, Message, EvaluateResult, final_text


def parse_pred(text: str) -> dict:
    """Pull the JSON object out of the model's answer (tolerates prose / code fences around it)."""
    if not text:
        return {}
    start, end = text.find("{"), text.rfind("}")
    if start == -1 or end == -1 or end < start:
        return {}
    try:
        obj = json.loads(text[start:end + 1])
        return obj if isinstance(obj, dict) else {}
    except Exception:
        return {}


def _norm(v):
    return str(v).strip().lower()


def build_eval_rows(examples, system_prompt):
    """EvaluationRows: system+user in, gold JSON string as ground_truth."""
    rows = []
    for ex in examples:
        gold = gold_label(ex)
        rows.append(EvaluationRow(
            messages=[Message(role="system", content=system_prompt),
                      Message(role="user", content=ex["prompt"])],
            ground_truth=json.dumps(gold),
        ))
    return rows


async def route_reward(row: EvaluationRow) -> EvaluationRow:
    pred = parse_pred(final_text(row))
    gold = json.loads(row.ground_truth)
    per = {f: int(f in pred and _norm(pred[f]) == _norm(gold[f])) for f in FIELDS}
    score = sum(per.values()) / len(FIELDS)
    row.evaluation_result = EvaluateResult(score=score, reason=json.dumps(per))
    return row


def run_eval(model, examples, label, system_prompt):
    """Run eval-protocol over the rows and return (exact_match, per_field_accuracy dict)."""
    # reasoning_effort="none": we trained on terse JSON labels with NO reasoning traces, so we
    # eval the same way — thinking off. qwen3.5 also ignores the "/no_think" tag and otherwise emits
    # a long "Thinking Process:" preamble that runs past max_tokens -> no JSON -> parse fails (rows
    # score 0). Applied to BOTH base and tuned so the comparison stays fair.
    mean, rows = single_turn_eval(build_eval_rows(examples, system_prompt), model, route_reward,
                                  max_tokens=MAX_TOKENS, extra_completion_params={"reasoning_effort": "none"})
    per_totals = {f: 0 for f in FIELDS}
    exact = 0
    for r in rows:
        per = json.loads(r.evaluation_result.reason)
        for f in FIELDS:
            per_totals[f] += per[f]
        exact += int(all(per[f] for f in FIELDS))
    n = len(rows)
    per_acc = {f: per_totals[f] / n for f in FIELDS}
    print(f"[{label}] exact-match {exact}/{n} = {exact / n:.1%} | field means (mean row score) {mean:.1%}")
    return exact / n, per_acc


def eval_deployed(model_resource, label, examples, system_prompt):
    """Deploy a model on-demand (dedicated), eval it, then DELETE the deployment.
    We score the base through the same on-demand deployment path as the tuned model so the
    before/after comparison is apples-to-apples on this managed (dedicated) notebook."""
    did = f"routereval-{label}-{uuid.uuid4().hex[:6]}".replace(" ", "").replace("(", "").replace(")", "")
    dep = client.deployments.create(base_model=model_resource, deployment_id=did,
                                    accelerator_type=ACCELERATOR_TYPE,
                                    min_replica_count=1, max_replica_count=1)
    d_id = dep.name.split("/")[-1]
    try:
        wait_until(lambda: client.deployments.get(d_id), {"READY"}, {"FAILED"},
                   f"deploy-{label}", every=15, tries=160)
        infer = f"accounts/{ACCOUNT_ID}/deployments/{d_id}"
        return run_eval(infer, examples, label, system_prompt)
    finally:
        try:
            client.deployments.delete(d_id, ignore_checks=True)
            print(f"[{label}] deployment {d_id} deleted")
        except Exception:  # noqa: BLE001 - fall back to scale-to-0 if delete is blocked
            try:
                client.deployments.scale(d_id, replica_count=0)
                print(f"[{label}] scaled to 0 (delete blocked): {d_id}")
            except Exception as e2:  # noqa: BLE001
                print(f"[{label}] teardown skip: {str(e2)[:100]}")

/Users/sinan/cookbook-casestudies/.venv/lib/python3.11/site-packages/eval_protocol/models.py:1156: PydanticDeprecatedSince20: Support for class-based `config` is deprecated, use ConfigDict instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.13/migration/
  class TaskDefinitionModel(BaseModel):


## 5. Evaluate the base model (before)

Baseline accuracy of the un-tuned base model on the held-out eval rows, using the **rich
prompt** (routing rule spelled out) — the base model's best-effort setup. This is the bar
the fine-tuned model has to match with only the lean prompt. To keep the before/after
comparison apples-to-apples on this managed (dedicated) notebook, this **deploys the base
on-demand, scores it, and tears it down** (spends a little GPU) via `eval_deployed` — the
same deployment path used for the tuned model.

In [5]:
# Base model gets the RICH prompt (rule spelled out) — best-effort prompt engineering.
# Deploy it on-demand, score it, and tear it down — same path as the tuned model.
base_exact, base_fields = eval_deployed(BASE_MODEL, "base-rich", eval_ex, RICH_PROMPT)

[deploy-base-rich] 001 state=CREATING
[deploy-base-rich] 002 state=CREATING
[deploy-base-rich] 003 state=CREATING
[deploy-base-rich] 004 state=CREATING
[deploy-base-rich] 005 state=CREATING
[deploy-base-rich] 006 state=CREATING
[deploy-base-rich] 007 state=CREATING
[deploy-base-rich] 008 state=CREATING
[deploy-base-rich] 009 state=CREATING
[deploy-base-rich] 010 state=CREATING
[deploy-base-rich] 011 state=CREATING
[deploy-base-rich] 012 state=CREATING
[deploy-base-rich] 013 state=CREATING
[deploy-base-rich] 014 state=CREATING
[deploy-base-rich] 015 state=CREATING
[deploy-base-rich] 016 state=CREATING
[deploy-base-rich] 017 state=CREATING
[deploy-base-rich] 018 state=CREATING
[deploy-base-rich] 019 state=CREATING
[deploy-base-rich] 020 state=CREATING
[deploy-base-rich] 021 state=CREATING
[deploy-base-rich] 022 state=CREATING
[deploy-base-rich] 023 state=CREATING
[deploy-base-rich] 024 state=CREATING
[deploy-base-rich] 025 state=CREATING
[deploy-base-rich] 026 state=CREATING
[deploy-base

## 6. Fine-tune the router (GO LIVE)

Upload the training file, launch a managed LoRA SFT job, and poll it to completion.
**This costs money and GPU quota.**

In [6]:
dataset_name = upload_dataset(DATASET_ID, TRAIN_FILE, fmt="CHAT")
print("dataset:", dataset_name)

# Optional W&B logging (enabled only if WANDB_API_KEY + WANDB_ENTITY are in .env).
_wandb = ({"enabled": True, "api_key": os.getenv("WANDB_API_KEY"), "entity": os.getenv("WANDB_ENTITY"),
           "project": os.getenv("WANDB_PROJECT", "sft-router-sdk")}
          if os.getenv("WANDB_API_KEY") and os.getenv("WANDB_ENTITY") else None)

job = client.supervised_fine_tuning_jobs.create(
    dataset=dataset_name,
    base_model=BASE_MODEL,
    output_model=f"accounts/{ACCOUNT_ID}/models/{OUTPUT_MODEL_ID}",  # SDK wants the full resource path
    epochs=EPOCHS,
    lora_rank=LORA_RANK,
    learning_rate=LEARNING_RATE,
    batch_size_samples=BATCH_SIZE_SAMPLES,
    eval_auto_carveout=True,
    **({"wandb_config": _wandb} if _wandb else {}),
)
job_id = job.name.split("/")[-1]
print("SFT job:", job.name)
wait_until(lambda: client.supervised_fine_tuning_jobs.get(job_id),
           {"JOB_STATE_COMPLETED"}, {"JOB_STATE_FAILED", "JOB_STATE_CANCELLED"}, "sft")
TUNED_MODEL_NAME = f"accounts/{ACCOUNT_ID}/models/{OUTPUT_MODEL_ID}"
print("Tuned model:", TUNED_MODEL_NAME)

[dataset] 001 state=READY
dataset: accounts/pyroworks/datasets/route-sft-a73198
SFT job: accounts/pyroworks/supervisedFineTuningJobs/ldhnfmy8
[sft] 001 state=JOB_STATE_RUNNING
[sft] 002 state=JOB_STATE_RUNNING
[sft] 003 state=JOB_STATE_RUNNING
[sft] 004 state=JOB_STATE_RUNNING
[sft] 005 state=JOB_STATE_RUNNING
[sft] 006 state=JOB_STATE_RUNNING
[sft] 007 state=JOB_STATE_RUNNING
[sft] 008 state=JOB_STATE_RUNNING
[sft] 009 state=JOB_STATE_RUNNING
[sft] 010 state=JOB_STATE_RUNNING
[sft] 011 state=JOB_STATE_RUNNING
[sft] 012 state=JOB_STATE_RUNNING
[sft] 013 state=JOB_STATE_RUNNING
[sft] 014 state=JOB_STATE_RUNNING
[sft] 015 state=JOB_STATE_RUNNING
[sft] 016 state=JOB_STATE_RUNNING
[sft] 017 state=JOB_STATE_RUNNING
[sft] 018 state=JOB_STATE_RUNNING
[sft] 019 state=JOB_STATE_RUNNING
[sft] 020 state=JOB_STATE_RUNNING
[sft] 021 state=JOB_STATE_RUNNING
[sft] 022 state=JOB_STATE_RUNNING
[sft] 023 state=JOB_STATE_RUNNING
[sft] 024 state=JOB_STATE_RUNNING
[sft] 025 state=JOB_STATE_RUNNING
[sft] 02

## 7. Deploy the tuned model

Put the fine-tuned weights behind an on-demand deployment (`min_replica_count=1`),
addressed as `accounts/<account>/deployments/<id>`.

In [7]:
dep = client.deployments.create(
    base_model=TUNED_MODEL_NAME,
    deployment_id=DEPLOYMENT_ID,
    accelerator_type=ACCELERATOR_TYPE,
    min_replica_count=1,
    max_replica_count=1,
)
dep_id = dep.name.split("/")[-1]
wait_until(lambda: client.deployments.get(dep_id), {"READY"}, {"FAILED"}, "deploy", every=15, tries=160)
INFER_MODEL = f"accounts/{ACCOUNT_ID}/deployments/{dep_id}"
print("Deployed. Infer id:", INFER_MODEL)

[deploy] 001 state=CREATING
[deploy] 002 state=CREATING
[deploy] 003 state=CREATING
[deploy] 004 state=CREATING
[deploy] 005 state=CREATING
[deploy] 006 state=CREATING
[deploy] 007 state=CREATING
[deploy] 008 state=CREATING
[deploy] 009 state=CREATING
[deploy] 010 state=CREATING
[deploy] 011 state=CREATING
[deploy] 012 state=CREATING
[deploy] 013 state=CREATING
[deploy] 014 state=CREATING
[deploy] 015 state=READY
Deployed. Infer id: accounts/pyroworks/deployments/router-sdk-f7c303


## 8. Evaluate the tuned model (after) & compare

Eval the tuned deployment with the **lean prompt** (no routing rule — it learned the policy)
and compare to the base's rich-prompt numbers. Headline is `route` accuracy (the actual
decision); exact-match + per-field are shown as secondary. Tuned matching or beating the
prompt-engineered base = the fine-tune paid off, on a shorter prompt.

In [8]:
# Tuned model gets the LEAN prompt (no rule) — it learned the policy during SFT.
tuned_exact, tuned_fields = run_eval(INFER_MODEL, eval_ex, "tuned (lean prompt)", LEAN_PROMPT)

rb, rt = base_fields[PRIMARY_FIELD], tuned_fields[PRIMARY_FIELD]
print("\n============== prompt-router: base (rich prompt) vs tuned (lean prompt) ==============")
print(f"  ROUTE accuracy (the decision):  {rb:.1%}  ->  {rt:.1%}   ({rt - rb:+.1%})   <- headline")
print(f"  exact-match (all {len(FIELDS)} fields):  {base_exact:.1%}  ->  {tuned_exact:.1%}"
      f"   ({tuned_exact - base_exact:+.1%})")
print("  per-field accuracy:")
print(f"    {'field':<12}{'base':>8}{'tuned':>8}{'delta':>8}")
for f in FIELDS:
    b, t = base_fields[f], tuned_fields[f]
    print(f"    {f:<12}{b:>7.0%}{t:>8.0%}{t - b:>+8.0%}")
print("=========================================================================")

[tuned (lean prompt)] exact-match 43/60 = 71.7% | field means (mean row score) 93.3%

============== prompt-router: base (rich prompt) vs tuned (lean prompt) ==============
  ROUTE accuracy (the decision):  90.0%  ->  98.3%   (+8.3%)   <- headline
  exact-match (all 5 fields):  25.0%  ->  71.7%   (+46.7%)
  per-field accuracy:
    field           base   tuned   delta
    complexity      52%     75%    +23%
    math            97%     97%     +0%
    code            97%    100%     +3%
    reasoning       50%     97%    +47%
    route           90%     98%     +8%


## Speed & cost on the holdout (dedicated)

Single-stream serving benchmark of the tuned deployment over the full holdout: wall-clock,
per-request latency, token throughput, and cost. Dedicated bills **per GPU-hour** while the
deployment is live, so the cost here is the GPU time to serve the set; the serverless
**per-token** equivalent is printed alongside for the comparison. The tuned deployment is
still live from the eval above — run this before the cleanup cell. (Base is the same model
size, so its serving speed/cost match the tuned model.)

In [ ]:
# Benchmark the tuned deployment over the full holdout via the OpenAI-compatible endpoint,
# capturing per-request latency + token usage. Sequential (single-stream) so latency and
# throughput are clean and comparable to the serverless run.
import time
import requests

INFER_URL = "https://api.fireworks.ai/inference/v1/chat/completions"
_HEADERS = {"Authorization": f"Bearer {os.environ['FIREWORKS_API_KEY']}", "Content-Type": "application/json"}


def _bench_one(messages):
    # reasoning_effort="none" mirrors run_eval (and the serverless disable-thinking eval) so
    # qwen3.5 emits JSON instead of long reasoning -- keeps latency/token/cost comparable, not inflated.
    body = {"model": INFER_MODEL, "messages": messages, "temperature": 0.0,
            "max_tokens": MAX_TOKENS, "reasoning_effort": "none"}
    t0 = time.time()
    r = requests.post(INFER_URL, headers=_HEADERS, json=body, timeout=180)
    dt = time.time() - t0
    r.raise_for_status()
    usage = r.json().get("usage", {}) or {}
    return dt, int(usage.get("prompt_tokens", 0)), int(usage.get("completion_tokens", 0))


def benchmark_dedicated(examples, system_prompt, label):
    lats, prompt_tok, out_tok = [], 0, 0
    t0 = time.time()
    for ex in examples:
        msgs = [{"role": "system", "content": system_prompt}, {"role": "user", "content": ex["prompt"]}]
        dt, p, c = _bench_one(msgs)
        lats.append(dt); prompt_tok += p; out_tok += c
    wall = time.time() - t0
    n = len(examples)
    gpu_cost = wall / 3600 * H200_HOURLY_USD * DEPLOY_REPLICAS            # actual dedicated billing
    token_equiv = prompt_tok / 1e6 * SERVERLESS_PREFILL_PER_1M + out_tok / 1e6 * SERVERLESS_SAMPLE_PER_1M
    res = {
        "n": n, "wall_s": wall, "req_per_s": n / wall, "avg_latency_s": sum(lats) / n,
        "prompt_tokens": prompt_tok, "output_tokens": out_tok, "out_tok_per_s": out_tok / wall,
        "gpu_cost_usd": gpu_cost, "cost_per_1k_req_usd": gpu_cost / n * 1000, "token_equiv_usd": token_equiv,
    }
    print(f"\n============== {label}: speed & cost on {n} holdout rows ==============")
    print(f"  wall-clock        : {wall:.1f}s   ({res['req_per_s']:.2f} req/s, avg {res['avg_latency_s']:.2f}s/req)")
    print(f"  tokens            : {prompt_tok:,} prompt + {out_tok:,} output   ({res['out_tok_per_s']:.0f} out tok/s)")
    print(f"  COST (dedicated)  : ${gpu_cost:.4f}  (GPU-hour: ${H200_HOURLY_USD:.2f}/hr x {DEPLOY_REPLICAS} replica)")
    print(f"                      -> ${res['cost_per_1k_req_usd']:.2f} per 1k requests")
    print(f"  token-equiv ref   : ${token_equiv:.4f}  (if billed at serverless per-token rates)")
    print("  note: dedicated also bills for provisioning + idle time until you tear it down.")
    print("=====================================================================")
    return res


managed_bench = benchmark_dedicated(eval_ex, LEAN_PROMPT, "dedicated tuned (qwen3p5-9b)")

## 9. Cleanup — tear down the deployment

Delete the deployment once you're done so it stops billing (safe to re-run).

In [10]:
# Fireworks blocks deleting a deployment that got traffic in the last hour; if delete is
# refused we scale it to 0 replicas (stops billing now; delete later after the cooldown).
def _teardown(dep_id):
    try:
        client.deployments.delete(dep_id, ignore_checks=True)  # bypass the recent-traffic guard
        print("deleted deployment:", dep_id)
    except Exception as e:  # noqa: BLE001
        try:
            client.deployments.scale(dep_id, replica_count=0)
            print(f"scaled to 0 (delete blocked): {dep_id}")
        except Exception as e2:  # noqa: BLE001
            print(f"skip {dep_id}: {str(e2)[:120]}")


for _dep in [globals().get("DEPLOYMENT_ID")]:
    if _dep:
        _teardown(_dep)

deleted deployment: router-sdk-f7c303
